In [11]:
import os
import re
import sys
import math
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)
sys.path.append(parent_dir + '/SERGIO')
sys.path.append(current_dir + '/vaes')
print(sys.path)

import yaml
import torch
import numpy as np
import torch.nn as nn
import seaborn as sns
from tqdm import tqdm
import networkx as nx
from torch import optim
from models.base import BaseVAE
from typing import List, Any
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import torch.nn.functional as F
from scipy.stats import ttest_ind
from sklearn.manifold import TSNE
from pytorch_lightning import Trainer
from scipy.sparse.linalg import bicgstab
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning import LightningDataModule
from scipy.spatial.distance import pdist, squareform
from pytorch_lightning.strategies import DDPStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from lightning_lite.utilities.seed import seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from sklearn.metrics import roc_auc_score, mean_squared_error, silhouette_score

from GENIE3.GENIE3 import *
nthreads=12

['/scratch/ab9738/dfdl_imputation/contrastive_learning', '/scratch/yz5944/miniconda3/envs/bio2/lib/python310.zip', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/lib-dynload', '', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages', '/scratch/ab9738/dfdl_imputation', '/scratch/ab9738/dfdl_imputation/SERGIO', '/scratch/ab9738/dfdl_imputation/contrastive_learning/vaes', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages/setuptools/_vendor', '/state/partition1/job-51393835/tmp0f5xn_kn', '/scratch/ab9738/dfdl_imputation', '/scratch/ab9738/dfdl_imputation/SERGIO', '/scratch/ab9738/dfdl_imputation/contrastive_learning/vaes']


In [12]:
def parse_dataset_name(folder_name):
    pattern1 = r'De-noised_(\d+)G_(\d+)T_(\d+)cPerT_dynamics_(\d+)_DS(\d+)'
    pattern2 = r'De-noised_(\d+)G_(\d+)T_(\d+)cPerT_(\d+)_DS(\d+)'
    match_p1 = re.match(pattern1, folder_name)
    match_p2 = re.match(pattern2, folder_name)
    if match_p1:
        return {
            'number_genes': int(match_p1.group(1)),
            'number_bins': int(match_p1.group(2)),
            'cells_per_type': int(match_p1.group(3)),
            'dynamics': int(match_p1.group(4)),
            'dataset_id': int(match_p1.group(5)),
            'folder_name': folder_name
        }
    if match_p2:
        return {
            'number_genes': int(match_p2.group(1)),
            'number_bins': int(match_p2.group(2)),
            'cells_per_type': int(match_p2.group(3)),
            'dynamics': int(match_p2.group(4)),
            'dataset_id': int(match_p2.group(5)),
            'folder_name': folder_name
        }
    return

def get_datasets():
    datasets = []
    data_sets_dir = '../SERGIO/data_sets'
    for folder_name in os.listdir(data_sets_dir):
        dataset_info = parse_dataset_name(folder_name)
        if dataset_info:
            datasets.append(dataset_info)
    return sorted(datasets, key=lambda x: x['dataset_id'])

def load_data(dataset_info):
    dataset_id = dataset_info['dataset_id']
    data_dir = f'../SERGIO/imputation_data_2/DS{dataset_id}/'
    ds_clean_path = os.path.join(data_dir, 'DS6_clean.npy')
    ds_noisy_path = os.path.join(data_dir, 'DS6_45_iter_0.npy')
    if not os.path.exists(ds_clean_path) or not os.path.exists(ds_noisy_path):
        print(f"Data files not found for Dataset {dataset_id}. Skipping.")
        return None, None
    ds_clean = np.load(ds_clean_path).astype(np.float32)
    ds_noisy = np.load(ds_noisy_path).astype(np.float32)
    return ds_clean, ds_noisy

def load_interactions_info(num_genes, interactions_file):
    gt = np.zeros((num_genes, num_genes))
    with open(interactions_file, 'r') as f:
        lines = f.readlines()
    for line in lines:
        line_list = line.strip().split(',')
        target_index = int(float(line_list[0]))
        num_regs = int(float(line_list[1]))
        for i in range(num_regs):
            try:
                reg_index = int(float(line_list[i + 2]))
                gt[reg_index, target_index] = 1
            except:
                continue
    return gt

### Get H and split into G where G = {train, valid} and H/G = {test}

In [13]:
def load_ground_truth_grn(file_path, num_genes):
    H = np.zeros((num_genes, num_genes))
    with open(file_path, 'r') as f:
        for line in f:
            source, target = map(int, line.strip().split(','))
            H[source, target] = 1
    return H

def sample_partial_grn(H, sample_ratio=8/10):
    num_edges = np.sum(H)
    num_sample = int(num_edges * sample_ratio)
    edge_indices = np.argwhere(H == 1)
    np.random.shuffle(edge_indices)
    sampled_edges = edge_indices[:num_sample]
    G = np.zeros_like(H)
    G[tuple(zip(*sampled_edges))] = 1
    return G

def split_train_valid(G, train_ratio=7/8):  # 7:1 ratio
    edge_indices = np.argwhere(G == 1)
    num_edges = len(edge_indices)
    num_train = int(num_edges * train_ratio)
    np.random.shuffle(edge_indices)
    train_edges = edge_indices[:num_train]
    valid_edges = edge_indices[num_train:]
    G_train = np.zeros_like(G)
    G_valid = np.zeros_like(G)
    G_train[tuple(zip(*train_edges))] = 1
    G_valid[tuple(zip(*valid_edges))] = 1
    return G_train, G_valid

def get_test_set(H, G):
    G_test = H - G
    G_test[G_test < 0] = 0
    return G_test

### Plot GRNs

In [14]:
def get_clusters_from_adj(adj):
    G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
    unclustered = set(G.nodes())
    cluster_labels = {}
    cluster_id = 0
    while unclustered:
        start_node = unclustered.pop()
        cluster = set()
        to_explore = {start_node}
        while to_explore:
            node = to_explore.pop()
            cluster.add(node)
            successors = set(G.successors(node)) - cluster
            to_explore.update(successors)
            predecessors = set(G.predecessors(node)) - cluster
            to_explore.update(predecessors)
        for node in cluster:
            cluster_labels[node] = cluster_id
            unclustered.discard(node)
        cluster_id += 1
    return cluster_labels

def get_cluster_labels(cluster_labels, num_genes):
    cluster_labels_list = np.full(num_genes, -1)
    for gene_idx in range(num_genes):
        if gene_idx in cluster_labels:
            cluster_labels_list[gene_idx] = cluster_labels[gene_idx]
    max_label = max(cluster_labels.values()) if cluster_labels else -1
    unassigned_label = max_label + 1
    cluster_labels_list[cluster_labels_list == -1] = unassigned_label
    return cluster_labels_list

def plot_grn_from_graphs(G_train, G_valid, G_test, H, dataset_id):
    num_genes = G_train.shape[0]
    cluster_labels_train = get_clusters_from_adj(G_train)
    cluster_labels_list_train = get_cluster_labels(cluster_labels_train, num_genes)
    cluster_labels_valid = get_clusters_from_adj(G_valid)
    cluster_labels_list_valid = get_cluster_labels(cluster_labels_valid, num_genes)
    cluster_labels_test = get_clusters_from_adj(G_test)
    cluster_labels_list_test = get_cluster_labels(cluster_labels_test, num_genes)
    cluster_labels_H = get_clusters_from_adj(H)
    cluster_labels_list_H = get_cluster_labels(cluster_labels_H, num_genes)

    plot_grn(G_train, cluster_labels_list_train, 'Training Set GRN', f'./results/cl/DS{dataset_id}/grn_train.png')
    plot_grn(G_valid, cluster_labels_list_valid, 'Validation Set GRN', f'./results/cl/DS{dataset_id}/grn_valid.png')
    plot_grn(G_test, cluster_labels_list_test, 'Test Set GRN', f'./results/cl/DS{dataset_id}/grn_test.png')
    plot_grn(H, cluster_labels_list_H, 'Full Ground Truth GRN', f'./results/cl/DS{dataset_id}/grn_full.png')

    return cluster_labels_list_train, cluster_labels_list_valid, cluster_labels_list_test, cluster_labels_list_H

def plot_grn(adj_matrix, cluster_labels, title, save_path):
    G = nx.from_numpy_array(adj_matrix, create_using=nx.DiGraph)
    pos = nx.spring_layout(G, seed=42)
    node_colors = cluster_labels[list(G.nodes())]
    plt.figure(figsize=(10, 10))
    nx.draw_networkx_nodes(G, pos, node_size=50,
                           cmap=plt.cm.get_cmap('nipy_spectral', int(np.max(node_colors)) + 1),
                           node_color=node_colors)
    nx.draw_networkx_edges(G, pos, alpha=0.5, arrows=True)
    plt.title(title)
    plt.axis('off')
    plt.savefig(save_path)
    plt.close()

def plot_embeddings(embeddings, cluster_labels, title, save_path):
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings)
    plt.figure(figsize=(10, 10))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                          cmap=plt.cm.get_cmap('nipy_spectral', int(np.max(cluster_labels))+1),
                          c=cluster_labels, s=50, alpha=0.7)
    plt.colorbar(scatter, label='Cluster Labels')
    plt.title(title)
    plt.axis('off')
    plt.savefig(save_path)
    plt.close()

# Contrastive Learning Algo

In [15]:
def get_balanced_edges(adj_matrix, num_batches):
    positive_edges = np.argwhere(adj_matrix == 1)
    negative_edges = np.argwhere(adj_matrix == 0)
    num_positive = len(positive_edges)
    num_negative = len(negative_edges)
    edges_per_batch = num_positive // num_batches
    
    np.random.shuffle(positive_edges)
    np.random.shuffle(negative_edges)
    
    balanced_edges = []
    for i in range(num_batches):
        batch_positive = positive_edges[i * edges_per_batch : (i + 1) * edges_per_batch]
        batch_negative = negative_edges[i * edges_per_batch : (i + 1) * edges_per_batch]
        balanced_edges.append(np.concatenate([batch_positive, batch_negative]))
    
    return balanced_edges

def edge_score(embedding_i, embedding_j):
    # Element-wise multiplication followed by summation
    print(f"shape of embedding_i: {embedding_i.shape}, embedding_j: {embedding_j.shape}")
    scores = torch.sum(embedding_i * embedding_j, dim=1)  # Shape: [batch_size]
    print(f"shape of scores: {scores.shape}")
    print(f"shape of torch.sigmoid(scores): {torch.sigmoid(scores).shape}")
    return torch.sigmoid(scores)  # Shape: [batch_size]

def compute_edge_likelihoods(embeddings):
    similarity_matrix = np.dot(embeddings, embeddings.T)
    probabilities = 1 / (1 + np.exp(-similarity_matrix))  # Sigmoid to map to [0, 1]
    return probabilities


# Define VAE dataset structure

In [16]:
class GRNDataset(torch.utils.data.Dataset):
    def __init__(self, data, adjacency_matrix):
        self.data = torch.FloatTensor(data)  # Shape: (400, 2700)
        self.adjacency_matrix = torch.FloatTensor(adjacency_matrix)  # Shape: (400, 400)

    def __len__(self):
        return self.data.shape[0]  # Number of genes

    def __getitem__(self, idx):
        return self.data[idx], self.adjacency_matrix[idx]

class GRNVAEDataset(LightningDataModule):
    def __init__(
        self,
        data: np.ndarray,
        adjacency_matrix: np.ndarray,
        train_val_test_split: tuple = (0.7, 0.15, 0.15),
        train_batch_size: int = 32,
        val_batch_size: int = 32,
        test_batch_size: int = 32,
        num_workers: int = 0,
        pin_memory: bool = False
    ):
        super().__init__()
        self.data = data
        self.adjacency_matrix = adjacency_matrix
        self.train_val_test_split = train_val_test_split
        self.train_batch_size = train_batch_size
        self.val_batch_size = val_batch_size
        self.test_batch_size = test_batch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory

    def setup(self, stage: str = None):
        # Create train, validation, and test splits
        num_samples = len(self.data)
        indices = np.random.permutation(num_samples)
        train_split, val_split, test_split = self.train_val_test_split
        train_end = int(train_split * num_samples)
        val_end = train_end + int(val_split * num_samples)

        train_indices = indices[:train_end]
        val_indices = indices[train_end:val_end]
        test_indices = indices[val_end:]

        self.train_dataset = GRNDataset(self.data[train_indices], self.adjacency_matrix[train_indices])
        self.val_dataset = GRNDataset(self.data[val_indices], self.adjacency_matrix[val_indices])
        self.test_dataset = GRNDataset(self.data[test_indices], self.adjacency_matrix[test_indices])

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.train_batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.val_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.test_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
        )

# VAE experiment

In [17]:
class GRNVAEExperiment(pl.LightningModule):
    def __init__(self, vae_model: BaseVAE, params: dict) -> None:
        super(GRNVAEExperiment, self).__init__()

        self.model = vae_model
        self.params = params
        self.curr_device = None

    def forward(self, input: torch.Tensor, **kwargs) -> torch.Tensor:
        return self.model(input, **kwargs)

    def training_step(self, batch, batch_idx, optimizer_idx=0):
        gene_expr, adj_matrix = batch
        self.curr_device = gene_expr.device

        results = self.forward(gene_expr)
        train_loss = self.model.loss_function(*results,
                                              M_N=self.params['kld_weight'],
                                              optimizer_idx=optimizer_idx,
                                              batch_idx=batch_idx)

        self.log_dict({key: val.item() for key, val in train_loss.items()}, sync_dist=True)

        return train_loss['loss']

    def validation_step(self, batch, batch_idx, optimizer_idx=0):
        gene_expr, adj_matrix = batch
        self.curr_device = gene_expr.device

        results = self.forward(gene_expr)
        val_loss = self.model.loss_function(*results,
                                            M_N=1.0,
                                            optimizer_idx=optimizer_idx,
                                            batch_idx=batch_idx)

        self.log_dict({f"val_{key}": val.item() for key, val in val_loss.items()}, sync_dist=True)

    def on_validation_end(self) -> None:
        self.sample_genes()

    def sample_genes(self):
        # Get sample reconstruction
        test_input, _ = next(iter(self.trainer.datamodule.test_dataloader()))
        test_input = test_input.to(self.curr_device)

        recons = self.model.generate(test_input)
        
        # Save reconstructions as numpy array
        np.save(os.path.join(self.logger.log_dir, 
                             "Reconstructions", 
                             f"recons_Epoch_{self.current_epoch}.npy"),
                recons.cpu().numpy())

        try:
            # Generate new samples
            samples = self.model.sample(100, self.curr_device)
            
            # Save samples as numpy array
            np.save(os.path.join(self.logger.log_dir, 
                                 "Samples", 
                                 f"samples_Epoch_{self.current_epoch}.npy"),
                    samples.cpu().numpy())
        except Warning:
            pass

    def configure_optimizers(self):
        optimizer = optim.Adam(self.model.parameters(),
                               lr=self.params['LR'],
                               weight_decay=self.params['weight_decay'])

        scheduler = optim.lr_scheduler.ExponentialLR(optimizer,
                                                     gamma=self.params['scheduler_gamma'])

        return [optimizer], [scheduler]

# VAE model

In [18]:

class GRNVanillaVAE(nn.Module):
    def __init__(self,
                 num_genes: int,
                 num_cells: int,
                 latent_dim: int,
                 hidden_dims: List = None,
                 **kwargs) -> None:
        super(GRNVanillaVAE, self).__init__()
        self.num_genes = num_genes
        self.num_cells = num_cells
        self.latent_dim = latent_dim
        if hidden_dims is None:
            hidden_dims = [512, 256, 128, 64, 32]

        # Build Encoder
        modules = []
        in_features = num_cells
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Linear(in_features, h_dim),
                    nn.BatchNorm1d(num_genes),
                    nn.LeakyReLU())
            )
            in_features = h_dim

        self.encoder = nn.Sequential(*modules)
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1], latent_dim)

        # Build Decoder
        modules = []

        self.decoder_input = nn.Linear(latent_dim, hidden_dims[-1])

        hidden_dims.reverse()

        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.Linear(hidden_dims[i], hidden_dims[i + 1]),
                    nn.BatchNorm1d(num_genes),
                    nn.LeakyReLU())
            )

        self.decoder = nn.Sequential(*modules)

        self.final_layer = nn.Linear(hidden_dims[-1], num_cells)

    def encode(self, input: torch.Tensor) -> List[torch.Tensor]:
        result = self.encoder(input)
        mu = self.fc_mu(result)
        log_var = self.fc_var(result)
        return [mu, log_var]

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        result = self.decoder_input(z)
        result = self.decoder(result)
        result = self.final_layer(result)
        return result

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, input: torch.Tensor, **kwargs) -> List[torch.Tensor]:
        mu, log_var = self.encode(input)
        z = self.reparameterize(mu, log_var)
        return [self.decode(z), input, mu, log_var]

    def loss_function(self, *args, **kwargs) -> dict:
        recons = args[0]
        input = args[1]
        mu = args[2]
        log_var = args[3]

        kld_weight = kwargs['M_N']
        recons_loss = F.mse_loss(recons, input)

        kld_loss = torch.mean(-0.5 * torch.sum(1 + log_var - mu ** 2 - log_var.exp(), dim=1), dim=0)

        loss = recons_loss + kld_weight * kld_loss
        return {'loss': loss, 'Reconstruction_Loss': recons_loss.detach(), 'KLD': -kld_loss.detach()}

    def sample(self, num_samples: int, current_device: int, **kwargs) -> torch.Tensor:
        z = torch.randn(num_samples, self.latent_dim)
        z = z.to(current_device)
        samples = self.decode(z)
        return samples

    def generate(self, x: torch.Tensor, **kwargs) -> torch.Tensor:
        return self.forward(x)[0]

# VAE Embeddings

In [19]:

from pathlib import Path

def VAE_embeddings(ds, G_train, G_valid, dataset_id, config_path):
    with open(config_path, 'r') as file:
        try:   
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)

    tb_logger = TensorBoardLogger(save_dir=config['logging_params']['save_dir'],
                                  name=f"{config['model_params']['name']}_DS{dataset_id}")

    seed_everything(config['exp_params']['manual_seed'], True)
    model = GRNVanillaVAE(num_genes=ds.shape[0], num_cells=ds.shape[1], **config['model_params'])
    experiment = GRNVAEExperiment(model, config['exp_params'])

    data = GRNVAEDataset(
        data=ds,
        adjacency_matrix=G_train,  # Use G_train as the adjacency matrix
        **config["data_params"], pin_memory=len(config['trainer_params']['gpus']) != 0
    )
    data.setup()

    runner = Trainer(logger=tb_logger,
                    callbacks=[
                        LearningRateMonitor(),
                        ModelCheckpoint(save_top_k=2, 
                                        dirpath =os.path.join(tb_logger.log_dir , "checkpoints"), 
                                        monitor= "val_loss",
                                        save_last= True),
                    ],
                    strategy=DDPStrategy(find_unused_parameters=False),
                    **config['trainer_params'])

    Path(f"{tb_logger.log_dir}/Samples").mkdir(exist_ok=True, parents=True)
    Path(f"{tb_logger.log_dir}/Reconstructions").mkdir(exist_ok=True, parents=True)

    print(f"======= Training VAE for Dataset {dataset_id} =======")
    runner.fit(experiment, datamodule=data)

    model.eval()
    with torch.no_grad():
        embeddings = model.encode(torch.tensor(ds, dtype=torch.float).to(model.device))[0]
        embeddings = embeddings.cpu().numpy()

    return embeddings

In [20]:
datasets = get_datasets()
# for dataset_info in datasets:
dataset_info = datasets[1]
# Main processing loop
dataset_id = dataset_info['dataset_id']
print(f"\nProcessing Dataset {dataset_id}...")
ds_clean, ds_noisy = load_data(dataset_info)

num_genes = ds_noisy.shape[0]
cells_per_type = dataset_info['cells_per_type']
num_cells = ds_noisy.shape[1]

gt_grn_file = f'../SERGIO/data_sets/{dataset_info["folder_name"]}/gt_GRN.csv'
H = load_ground_truth_grn(gt_grn_file, num_genes)
G = sample_partial_grn(H, sample_ratio=0.9)
G_train, G_valid = split_train_valid(G, train_ratio=8/9)  # 8:1 ratio
G_test = get_test_set(H, G)

embeddings = VAE_embeddings(ds_clean, G_train, G_valid, dataset_id, config_path='./vaes/configs/vae.yaml')

[rank: 0] Global seed set to 1265



Processing Dataset 2...


/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:441: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages/lightning_lite/plugins/environments/slurm.py:167: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/ ...
  rank_zero_warn(


MisconfigurationException: `Trainer(strategy='ddp')` is not compatible with an interactive environment. Run your code as a script, or choose one of the compatible strategies: Trainer(strategy=None|dp|ddp_fork). In case you are spawning processes yourself, make sure to include the Trainer creation inside the worker function.

# Contrastive Learning Algo

# Experiment

In [ ]:
def compute_validation_loss(model, ds_tensor, w_ij_valid, temperature):
    model.eval()
    with torch.no_grad():
        recon_data, _, _, z = model(ds_tensor)
        # Compute distances
        dot_product = torch.matmul(z, z.T)
        square_sum = torch.sum(z ** 2, dim=1, keepdim=True)
        distances = square_sum + square_sum.T - 2 * dot_product
        distances = torch.clamp(distances, min=0.0)
        # Compute contrastive loss using w_ij_valid
        numerator = torch.exp(-distances / temperature) * w_ij_valid
        denominator = torch.exp(-distances / temperature)
        loss_matrix = -torch.log((torch.sum(numerator, dim=1) / torch.sum(denominator, dim=1)) + 1e-8)
        contrastive_loss = torch.mean(loss_matrix)
    return contrastive_loss.item()

def run_pipeline(ds_noisy, ds_clean, interactions, G_train, G_valid, G_test, G, H, dataset_id):
    ds_imputed, embeddings = contrastive_imputation(ds_noisy, G_train, G_valid, dataset_id)

    mse = mean_squared_error(ds_clean.flatten(), ds_imputed.flatten())
    print(f"MSE between Clean and Imputed Data: {mse:.4f}")
    log_file.write(f"MSE between Clean and Imputed Data: {mse:.4f}\n")

    ds_imputed_T = ds_imputed.T
    
    VIM_imputed = GENIE3(ds_imputed_T, nthreads=12, ntrees=100, regulators='all', gene_names=[str(s) for s in range(ds_imputed_T.shape[1])])
    
    inferred_grn = VIM_imputed.flatten()
    
    roc_auc_train = roc_auc_score(G_train.flatten(), inferred_grn)
    roc_auc_valid = roc_auc_score(G_valid.flatten(), inferred_grn)
    roc_auc_test = roc_auc_score(G_test.flatten(), inferred_grn)
    roc_auc_total = roc_auc_score(G.flatten(), inferred_grn)
    roc_auc_total_traditional = roc_auc_score(interactions.flatten(), inferred_grn)
    
    print(f"ROC AUC Score on Training Set: {roc_auc_train:.4f}")
    print(f"ROC AUC Score on Validation Set: {roc_auc_valid:.4f}")
    print(f"ROC AUC Score on Test Set: {roc_auc_test:.4f}")
    print(f"ROC AUC Score on Total G (Train + Valid): {roc_auc_total:.4f}\n")
    print(f"ROC AUC Score on Total G (Train + Valid) using traditional GENIE3: {roc_auc_total_traditional:.4f}\n")
    
    log_file.write(f"ROC AUC Score on Training Set: {roc_auc_train:.4f}\n")
    log_file.write(f"ROC AUC Score on Validation Set: {roc_auc_valid:.4f}\n")
    log_file.write(f"ROC AUC Score on Test Set: {roc_auc_test:.4f}\n")
    log_file.write(f"ROC AUC Score on Total G (Train + Valid): {roc_auc_total:.4f}\n")
    log_file.write(f"ROC AUC Score on Total G (Train + Valid) using traditional GENIE3: {roc_auc_total_traditional:.4f}\n")
    
    roc_auc_full = roc_auc_score(H.flatten(), inferred_grn)
    print(f"ROC AUC Score on Full Ground Truth H: {roc_auc_full:.4f}\n")
    log_file.write(f"ROC AUC Score on Full Ground Truth H: {roc_auc_full:.4f}\n")

    results = {
        'roc_auc_train': roc_auc_train,
        'roc_auc_valid': roc_auc_valid,
        'roc_auc_test': roc_auc_test,
        'roc_auc_total': roc_auc_total,
        'roc_auc_full': roc_auc_full,
        'roc_auc_total_traditional': roc_auc_total_traditional,
        'mse': mse,
        'embeddings': embeddings
    }
    return results

In [ ]:


target_file = f'../SERGIO/data_sets/{dataset_info["folder_name"]}/Interaction_cID_{dataset_info["dynamics"]}.txt'
interactions = load_interactions_info(num_genes, target_file)

log_dir = f'./results/cl/DS{dataset_id}'
os.makedirs(log_dir, exist_ok=True)
log_file_path = f'./results/cl/DS{dataset_id}/log.txt'

cluster_labels_train, cluster_labels_valid, cluster_labels_test, cluster_labels_H = plot_grn_from_graphs(G_train, G_valid, G_test, H, dataset_id)

with open(log_file_path, 'w') as log_file:
    # Evaluate clean data
    print("Evaluating Clean Data...")
    ds_clean_T = ds_clean.T
    VIM_clean = GENIE3(ds_clean_T, nthreads=12, ntrees=100, regulators='all',
                    gene_names=[str(s) for s in range(ds_clean_T.shape[1])])
    roc_auc_clean = roc_auc_score(interactions.flatten(), VIM_clean.flatten())
    print(f"ROC AUC Score for Clean Data: {roc_auc_clean:.4f}\n")
    log_file.write(f"ROC AUC Score for Clean Data: {roc_auc_clean:.4f}\n")

    # Evaluate noisy dataƒ
    print("Evaluating Noisy Data...")
    ds_noisy_T = ds_noisy.T
    VIM_noisy = GENIE3(ds_noisy_T, nthreads=12, ntrees=100, regulators='all',
                    gene_names=[str(s) for s in range(ds_noisy_T.shape[1])])
    roc_auc_noisy = roc_auc_score(interactions.flatten(), VIM_noisy.flatten())
    print(f"ROC AUC Score for Noisy Data: {roc_auc_noisy:.4f}\n")
    log_file.write(f"ROC AUC Score for Noisy Data: {roc_auc_noisy:.4f}\n")

    # Compute MSE between noisy data and clean data
    mse_noisy = mean_squared_error(ds_clean.flatten(), ds_noisy.flatten())
    print(f"MSE between Noisy Data and Clean Data: {mse_noisy:.4f}\n")
    log_file.write(f"MSE between Noisy Data and Clean Data: {mse_noisy:.4f}\n")

    print("Running Imputation and Analysis Pipeline...")
    results = run_pipeline(ds_noisy, ds_clean, interactions, G_train, G_valid, G_test, G, H, dataset_id)
    embeddings = results['embeddings']
    
    edge_probabilities = compute_edge_likelihoods(embeddings)
    valid_edge_indices = np.argwhere(G_valid == 1)
    test_edge_indices = np.argwhere(G_test == 1)
    valid_connected_probs = edge_probabilities[valid_edge_indices[:, 0], valid_edge_indices[:, 1]]
    valid_non_connected_probs = edge_probabilities[G_valid == 0]
    test_connected_probs = edge_probabilities[test_edge_indices[:, 0], test_edge_indices[:, 1]]
    test_non_connected_probs = edge_probabilities[G_test == 0]
    avg_valid_connected_prob = np.mean(valid_connected_probs)
    avg_valid_non_connected_prob = np.mean(valid_non_connected_probs)
    avg_test_connected_prob = np.mean(test_connected_probs)
    avg_test_non_connected_prob = np.mean(test_non_connected_probs)
    print(f"Validation Set - Connected Avg Prob: {avg_valid_connected_prob:.4f}, Non-Connected Avg Prob: {avg_valid_non_connected_prob:.4f}")
    print(f"Test Set - Connected Avg Prob: {avg_test_connected_prob:.4f}, Non-Connected Avg Prob: {avg_test_non_connected_prob:.4f}")
    t_stat_valid, p_value_valid = ttest_ind(valid_connected_probs, valid_non_connected_probs)
    print(f"Validation Set - T-test: t-statistic = {t_stat_valid:.4f}, p-value = {p_value_valid:.4e}")
    t_stat_test, p_value_test = ttest_ind(test_connected_probs, test_non_connected_probs)
    print(f"Test Set - T-test: t-statistic = {t_stat_test:.4f}, p-value = {p_value_test:.4e}")
    log_file.write(f"Validation Set - Connected Avg Prob: {avg_valid_connected_prob:.4f}, Non-Connected Avg Prob: {avg_valid_non_connected_prob:.4f}\n")
    log_file.write(f"Validation Set - T-test: t-statistic = {t_stat_valid:.4f}, p-value = {p_value_valid:.4e}\n")
    log_file.write(f"Test Set - Connected Avg Prob: {avg_test_connected_prob:.4f}, Non-Connected Avg Prob: {avg_test_non_connected_prob:.4f}\n")
    log_file.write(f"Test Set - T-test: t-statistic = {t_stat_test:.4f}, p-value = {p_value_test:.4e}\n")

    log_file.write(f"MSE between Clean and Imputed Data: {results['mse']:.4f}\n")
    print("Summary of ROC AUC Scores:")
    print(f"Clean Data: {roc_auc_clean:.4f}")
    print(f"Noisy Data: {roc_auc_noisy:.4f}")
    print(f"Imputed Data - Training Set: {results['roc_auc_train']:.4f}")
    print(f"Imputed Data - Validation Set: {results['roc_auc_valid']:.4f}")
    print(f"Imputed Data - Test Set: {results['roc_auc_test']:.4f}")
    print(f"Imputed Data - Total G (Train + Valid): {results['roc_auc_total']:.4f}")
    print(f"Imputed Data - Full Ground Truth H: {results['roc_auc_full']:.4f}")
    print(f"Total G (Train + Valid) using traditional GENIE3: {results['roc_auc_total_traditional']:.4f}")
    log_file.write("\nSummary of ROC AUC Scores:\n")
    log_file.write(f"Clean Data: {roc_auc_clean:.4f}\n")
    log_file.write(f"Noisy Data: {roc_auc_noisy:.4f}\n")
    log_file.write(f"Imputed Data - Training Set: {results['roc_auc_train']:.4f}\n")
    log_file.write(f"Imputed Data - Validation Set: {results['roc_auc_valid']:.4f}\n")
    log_file.write(f"Imputed Data - Test Set: {results['roc_auc_test']:.4f}\n")
    log_file.write(f"Imputed Data - Total G (Train + Valid): {results['roc_auc_total']:.4f}\n")
    log_file.write(f"Imputed Data - Full Ground Truth H: {results['roc_auc_full']:.4f}\n")
    log_file.write(f"Total G (Train + Valid) using traditional GENIE3: {results['roc_auc_total_traditional']:.4f}\n")

    print("\nSummary of MSE Values:")
    print(f"Noisy Data vs. Clean Data: {mse_noisy:.4f}")
    print(f"Imputed Data vs. Clean Data: {results['mse']:.4f}")
    log_file.write("\nSummary of MSE Values:\n")
    log_file.write(f"Noisy Data vs. Clean Data: {mse_noisy:.4f}\n")
    log_file.write(f"Imputed Data vs. Clean Data: {results['mse']:.4f}\n")

    print("Analyzing embeddings to see if connected nodes are closer...")
    pairwise_distances = squareform(pdist(embeddings, metric='euclidean'))
    connected_indices = np.argwhere(H == 1)
    non_connected_indices = np.argwhere(H == 0)

    # num_samples = 10000
    np.random.shuffle(connected_indices)
    np.random.shuffle(non_connected_indices)
    # connected_indices = connected_indices[:num_samples]
    # non_connected_indices = non_connected_indices[:num_samples]

    connected_distances = pairwise_distances[connected_indices[:, 0], connected_indices[:, 1]]
    non_connected_distances = pairwise_distances[non_connected_indices[:, 0], non_connected_indices[:, 1]]
    plt.figure(figsize=(10, 6))
    sns.kdeplot(connected_distances, label='Connected Nodes')
    sns.kdeplot(non_connected_distances, label='Non-Connected Nodes')
    plt.title('Distance Distribution between Node Embeddings')
    plt.xlabel('Euclidean Distance')
    plt.ylabel('Density')
    plt.legend()
    plt.savefig(f'./results/cl/DS{dataset_id}/distance_distribution.png')
    plt.show()

    t_stat, p_value = ttest_ind(connected_distances, non_connected_distances)
    print(f"T-test: t-statistic = {t_stat:.4f}, p-value = {p_value:.4e}")
    log_file.write(f"T-test: t-statistic = {t_stat:.4f}, p-value = {p_value:.4e}\n")

    print("Performing t-SNE on embeddings...")
    plot_embeddings(embeddings, cluster_labels=cluster_labels_train, title='t-SNE Embeddings', save_path=f'./results/cl/DS{dataset_id}/tsne_embeddings.png')

    valid_indices = cluster_labels_H != -1
    valid_embeddings = embeddings[valid_indices]
    valid_labels = cluster_labels_H[valid_indices]
    unique_labels = np.unique(valid_labels)
    print(f"Unique Labels: {unique_labels}")
    print(f"Number of Unique Labels: {len(unique_labels)}")

    cluster_labels_H = get_clusters_from_adj(H)
    print(f"Cluster labels (cluster_labels_H): {cluster_labels_H}")
    cluster_labels_list_H = get_cluster_labels(cluster_labels_H, num_genes)
    num_clusters = len(set(cluster_labels_H.values()))
    print(f"Number of clusters: {num_clusters}")
    # sil_score = silhouette_score(valid_embeddings, valid_labels)
    # print(f"Silhouette Score: {sil_score:.4f}")
    # log_file.write(f"Silhouette Score: {sil_score:.4f}\n")